<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Impute Missing Values**


Estimated time needed: **30** minutes


In this lab, you will practice essential data wrangling techniques using the Stack Overflow survey dataset. The primary focus is on handling missing data and ensuring data quality. You will:

- **Load the Data:** Import the dataset into a DataFrame using the pandas library.

- **Clean the Data:** Identify and remove duplicate entries to maintain data integrity.

- **Handle Missing Values:** Detect missing values, impute them with appropriate strategies, and verify the imputation to create a complete and reliable dataset for analysis.

This lab equips you with the skills to effectively preprocess and clean real-world datasets, a crucial step in any data analysis project.


## Objectives


In this lab, you will perform the following:


-   Identify missing values in the dataset.

-   Apply techniques to impute missing values in the dataset.
  
-   Use suitable techniques to normalize data in the dataset.


-----


#### Install needed library


In [3]:
!pip install pandas

### Step 1: Import Required Libraries


In [4]:
import pandas as pd

### Step 2: Load the Dataset Into a Dataframe


#### **Read Data**
<p>
The functions below will download the dataset into your browser:
</p>


In [6]:
file_path ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UDKAZw-kz18Yj8P6icf_qw/survey-data-duplicates.csv"
df = pd.read_csv(file_path)

# Display the first few rows to ensure it loaded correctly
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

In [14]:
df.shape

(65447, 114)

### Step 3. Finding and Removing Duplicates
##### Task 1: Identify duplicate rows in the dataset.


In [8]:
## Write your code here
Duplicated_Rows=df[df.duplicated()]
print(Duplicated_Rows)

       ResponseId                                         MainBranch  \
65437           1                     I am a developer by profession   
65438           2                     I am a developer by profession   
65439           3                     I am a developer by profession   
65440           4                              I am learning to code   
65441           5                     I am a developer by profession   
65442           6                        I code primarily as a hobby   
65443           7  I am not primarily a developer, but I write co...   
65444           8                              I am learning to code   
65445           9                        I code primarily as a hobby   
65446          10                     I am a developer by profession   

                      Age                                         Employment  \
65437  Under 18 years old                                Employed, full-time   
65438     35-44 years old                      

##### Task 2: Remove the duplicate rows from the dataframe.



In [12]:
## Write your code here
Clean_df=df.drop_duplicates().reset_index(drop=True)
print(Clean_df)

       ResponseId                      MainBranch                 Age  \
0               1  I am a developer by profession  Under 18 years old   
1               2  I am a developer by profession     35-44 years old   
2               3  I am a developer by profession     45-54 years old   
3               4           I am learning to code     18-24 years old   
4               5  I am a developer by profession     18-24 years old   
...           ...                             ...                 ...   
65432       65433  I am a developer by profession     18-24 years old   
65433       65434  I am a developer by profession     25-34 years old   
65434       65435  I am a developer by profession     25-34 years old   
65435       65436  I am a developer by profession     18-24 years old   
65436       65437     I code primarily as a hobby     18-24 years old   

                Employment                            RemoteWork   Check  \
0      Employed, full-time                     

### Step 4: Finding Missing Values
##### Task 3: Find the missing values for all columns.


In [15]:
## Write your code here

Missing_Values = df.isnull().sum()
print(Missing_Values)


ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10635
                       ...  
JobSatPoints_11        36001
SurveyLength            9257
SurveyEase              9201
ConvertedCompYearly    42012
JobSat                 36321
Length: 114, dtype: int64


##### Task 4: Find out how many rows are missing in the column RemoteWork.


In [16]:
## Write your code here
missing_rows=df['RemoteWork'].isnull().sum()
print(missing_rows)

10635


### Step 5. Imputing Missing Values
##### Task 5: Find the value counts for the column RemoteWork.


In [17]:
## Write your code here

value_counts = df['RemoteWork'].value_counts()
print(value_counts)


RemoteWork
Hybrid (some remote, some in-person)    23015
Remote                                  20836
In-person                               10961
Name: count, dtype: int64


##### Task 6: Identify the most frequent (majority) value in the RemoteWork column.



In [22]:
## Write your code here
most_frequent=df['RemoteWork'].mode()[0]
print(most_frequent)

Hybrid (some remote, some in-person)


##### Task 7: Impute (replace) all the empty rows in the column RemoteWork with the majority value.



In [23]:
## Write your code here
df['RemoteWork'] = df['RemoteWork'].fillna("Hybrid")
print(df['RemoteWork'].isnull().sum())

0


##### Task 8: Check for any compensation-related columns and describe their distribution.



In [36]:
## Write your code here

comp_cols = [col for col in df.columns 
             if any(word in col.lower() for word in ['salary', 'pay', 'bonus', 'comp', 'wage', 'income'])]

print(comp_cols)



['CompTotal', 'AIComplex', 'ConvertedCompYearly']


In [41]:
#columns identified
cols = ['CompTotal', 'AIComplex', 'ConvertedCompYearly']

# Stats and replacing missing values
dist = df[cols].describe().T
dist['missing'] = df[cols].isna().sum()
dist['missing_%'] = (df[cols].isna().mean() * 100).round(2)

print(dist)

# Skewness positive skew → long right tail (very common for salary data)
# negative skew → long left tail
print("\nSkewness:")
print(df[cols].skew())

# Outliers
q1 = df[cols].quantile(0.25)
q3 = df[cols].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("\nOutliers (IQR method):")
print(((df[cols] < lower) | (df[cols] > upper)).sum())




                       count           mean            std  min      25%  \
CompTotal            33740.0  2.963841e+145  5.444117e+147  0.0  60000.0   
ConvertedCompYearly  23435.0   8.615529e+04   1.867570e+05  1.0  32712.0   

                          50%       75%            max  
CompTotal            110000.0  250000.0  1.000000e+150  
ConvertedCompYearly   65000.0  107971.5   1.625660e+07  


### Summary 


**In this lab, you focused on imputing missing values in the dataset.**

- Use the <code>pandas.read_csv()</code> function to load a dataset from a CSV file into a DataFrame.

- Download the dataset if it's not available online and specify the correct file path.



<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-11-05|1.3|Madhusudhan Moole|Updated lab|
|2024-10-29|1.2|Madhusudhan Moole|Updated lab|
|2024-09-27|1.1|Madhusudhan Moole|Updated lab|
|2024-09-26|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
